# 🏁 Template 4 : évaluation et conclusion

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Aucun code de solution ici : c'est toi qui écris.*

## Les 4 templates du projet

| # | Phase | Notebook | Entrée | Sortie |
|---|---|---|---|---|
| 1 | Préparation des données | `02-Template_1_preparation_donnees` | `src/spam.csv` | `src/train.csv`, `val.csv`, `test.csv` |
| 2 | Modèle simple : embeddings + petit réseau | `03-Template_2_modele_embeddings` | les 3 CSV | `scores/scores_embeddings.json`, `scores/predictions_embeddings.csv` |
| 3 | Transfer learning : DistilBERT | `04-Template_3_transfer_learning` | les 3 CSV | `scores/scores_transfer.json`, `scores/predictions_transfer.csv` |
| **4** | **Évaluation et conclusion** (ce notebook) | `05-Template_4_evaluation_conclusion` | les fichiers de `scores/` | le tableau final du livrable |

L'énoncé demande deux choses : un notebook qui fait le preprocessing et entraîne des modèles (templates 1 à 3), et **« State the achieved performance clearly »**. C'est l'objet de ce notebook : comparer les modèles **honnêtement**, comprendre leurs erreurs, et **recommander** une solution à AT&T.

Il ne réentraîne rien : il lit seulement les fichiers du dossier `scores/`. Il peut donc être fait par un autre membre du groupe, après un simple `git pull`.

## Comment s'en servir

Même rythme qu'au template 1 :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants (parfois avec une petite démo dans un autre contexte) ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie ton travail. ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés sont ceux qu'attendent les contrôles. Lance toujours les cellules **dans l'ordre**.

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

DOSSIER_SRC = Path("src")
DOSSIER_SCORES = Path("scores")
METRIQUES = ["accuracy", "precision", "recall", "f1"]

pd.set_option("display.max_colwidth", None)
print("fichiers disponibles :", sorted(p.name for p in DOSSIER_SCORES.glob("*")))

---
# Étape 1 — Rassembler les scores

🎯 Lire tous les fichiers `scores/scores_*.json` et les mettre dans un seul tableau `tableau`, **une ligne par modèle**.

Ajouter une colonne `cle` : la fin du nom de fichier (`scores_embeddings.json` → `"embeddings"`). Elle permettra de retrouver le fichier de prédictions correspondant (`predictions_embeddings.csv`).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `sorted(DOSSIER_SCORES.glob("scores_*.json"))` | la liste des fichiers qui correspondent au motif | `*` = n'importe quoi |
| `chemin.stem` | le nom du fichier sans l'extension (`"scores_embeddings"`) | — |
| `texte.removeprefix("scores_")` | enlève un préfixe | — |
| `json.load(fichier)` | JSON → dictionnaire | — |
| `pd.DataFrame(liste_de_dictionnaires)` | une ligne par dictionnaire | — |

In [ ]:
# ✍️ À toi
# 1. pour chaque fichier scores_*.json : charger le dictionnaire et lui ajouter la clé "cle"
# 2. tableau = le DataFrame de ces dictionnaires ; affiche-le

In [ ]:
# ✅ Contrôle
assert {"embeddings", "transfer"} <= set(tableau["cle"]), "il manque un modèle : as-tu fait les templates 2 et 3 (ou un git pull) ?"
assert {"modele", "seuil", *METRIQUES, "n_parametres", "temps_entrainement_s", "cle"} <= set(tableau.columns)
print("✅ étape 1")

---
# Étape 2 — Ajouter la référence naïve

🎯 Un score n'a de sens que comparé à quelque chose. La comparaison minimale : un « modèle » qui répond **toujours ham**, sans rien apprendre. Calculer ses 4 métriques **sur test** et l'ajouter au tableau.

| colonne | valeur |
|---|---|
| `modele` | `"Référence : toujours ham"` |
| `cle` | `"naif"` |
| `seuil` | `np.nan` (« pas de valeur » : ce modèle n'a pas de seuil) |
| `n_parametres`, `temps_entrainement_s` | `0` |
| les 4 métriques | calculées avec des prédictions toutes à 0 |

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `np.zeros(n, dtype=int)` | n prédictions « ham » | — |
| `precision_score(..., zero_division=0)` | aucun spam prédit → précision non définie, on met 0 | `zero_division=0` |
| `pd.concat([tableau, pd.DataFrame([ligne])], ignore_index=True)` | ajoute une ligne | `ignore_index=True` |

In [ ]:
# ✍️ À toi
# 1. test_df = src/test.csv ; preds_naif = 776 zéros
# 2. naif = le dictionnaire de la référence (toutes les colonnes de tableau)
# 3. tableau = tableau + cette ligne

In [ ]:
# ✅ Contrôle
ligne = tableau[tableau["cle"] == "naif"]
assert len(ligne) == 1, "la référence doit apparaître une fois (relance depuis l'étape 1 si tu l'as ajoutée deux fois)"
assert round(ligne["accuracy"].iloc[0], 4) == 0.8737 and ligne["f1"].iloc[0] == 0
print("✅ étape 2")

👀 **Observe** : 87 % d'accuracy pour un modèle qui ne détecte **aucun** spam. C'est l'argument à donner à l'oral pour expliquer pourquoi on juge sur le F1 et le recall.

---
# Étape 3 — Le tableau final et son graphique

🎯 Le tableau du livrable : trié du meilleur au moins bon **F1**, avec les métriques arrondies. Puis un graphique en barres groupées : précision, recall et F1 de chaque modèle côte à côte.

`px.bar` veut des données **longues** (une ligne par couple modèle × métrique), alors que `tableau` est **large** (une colonne par métrique) : `melt` fait la conversion.

```
large :  modele | precision | recall          long :  modele | metrique  | valeur
         A      | 0.90      | 0.80     melt→          A      | precision | 0.90
                                                      A      | recall    | 0.80
```

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.sort_values("f1", ascending=False)` | trie | `ascending=False` : du plus grand au plus petit |
| `.round(3)` | arrondit | — |
| `df.melt(id_vars=..., value_vars=..., var_name=..., value_name=...)` | large → long | `id_vars="modele"` |
| `px.bar(long, x="modele", y="valeur", color="metrique", barmode="group")` | barres groupées | `barmode="group"`, `text_auto=".2f"` |

In [ ]:
# ✍️ À toi
# 1. tableau = trié par f1 décroissant, index renuméroté ; affiche modele + METRIQUES arrondis à 3 chiffres
# 2. long = tableau mis en format long (precision, recall, f1)
# 3. le graphique en barres groupées

In [ ]:
# ✅ Contrôle
assert tableau["f1"].is_monotonic_decreasing, "le tableau doit être trié par F1 décroissant"
assert tableau.iloc[-1]["cle"] == "naif"
assert len(long) == 3 * len(tableau) and {"modele", "metrique", "valeur"} <= set(long.columns)
print(f"✅ étape 3 — meilleur F1 : {tableau.iloc[0]['modele']} ({tableau.iloc[0]['f1']:.3f})")

---
# Étape 4 — Les matrices de confusion

🎯 Pour chaque vrai modèle (`embeddings`, `transfer`) : relire ses prédictions, appliquer **son** seuil (colonne `seuil` du tableau), et afficher sa matrice de confusion. La ranger dans un dictionnaire `matrices` (clé → matrice).

```
                 prédit ham          prédit spam
vrai ham     [ vrais négatifs    |  FAUX POSITIFS  ]   ← SMS légitimes bloqués à tort
vrai spam    [ FAUX NÉGATIFS     |  vrais positifs ]   ← spams qui passent
```

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `pd.read_csv(DOSSIER_SCORES / f"predictions_{cle}.csv")` | les prédictions d'un modèle | — |
| `tableau.set_index("cle").loc[cle, "seuil"]` | le seuil d'un modèle | — |
| `confusion_matrix(vrais, predits)` | la matrice 2 × 2 | — |
| `px.imshow(matrice, text_auto=True, x=[...], y=[...])` | la matrice en image, avec les nombres | `x=["prédit ham", "prédit spam"]`, `y=["vrai ham", "vrai spam"]` |

In [ ]:
# ✍️ À toi
# 1. predictions = {} et matrices = {}
# 2. pour chaque cle dans ["embeddings", "transfer"] :
#    - predictions[cle] = le DataFrame de predictions_{cle}.csv, avec une colonne "pred" (proba >= seuil du modèle, en 0/1)
#    - matrices[cle] = sa matrice de confusion ; affiche-la avec px.imshow (titre = nom du modèle)

In [ ]:
# ✅ Contrôle
for cle, m in matrices.items():
    assert m.shape == (2, 2) and m.sum() == 776 and m[1].sum() == 98, f"{cle} : matrice incohérente"
    recall = m[1, 1] / m[1].sum()
    attendu = tableau.set_index("cle").loc[cle, "recall"]
    assert abs(recall - attendu) < 1e-9, f"{cle} : le recall de la matrice ({recall:.3f}) ≠ celui du JSON ({attendu:.3f}) — as-tu pris le bon seuil ?"
print("✅ étape 4")

✍️ **À écrire** : traduis les matrices pour AT&T, en langage courant. Par exemple : « sur 678 SMS légitimes, le modèle X en bloque N à tort ; sur 98 spams, il en laisse passer M ». Lequel de ces deux nombres compte le plus pour les clients ?

*Ta réponse :*

---
# Étape 5 — Performance contre coût

🎯 Le meilleur F1 n'est pas tout : un modèle sert **en production**, sur des millions de SMS par jour. Comparer la taille et la durée d'entraînement des deux modèles : combien de **fois** plus gros et plus lent est DistilBERT, pour combien de points de F1 en plus ?

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `tableau.set_index("cle").loc[cle, colonne]` | une valeur d'un modèle | — |
| `px.bar(..., log_y=True)` | échelle logarithmique, indispensable quand les valeurs vont de 100 000 à 66 000 000 | `log_y=True` |

In [ ]:
# ✍️ À toi
# 1. ratio_parametres = n_parametres de transfer / celui d'embeddings ; ratio_temps = idem pour temps_entrainement_s
# 2. gain_f1 = f1 de transfer - f1 d'embeddings (en points : × 100)
# 3. affiche les trois, puis un graphique en barres du nombre de paramètres des 2 modèles (échelle log)

In [ ]:
# ✅ Contrôle
assert ratio_parametres > 100
assert ratio_temps > 1
assert -1 < gain_f1 < 1
print("✅ étape 5")

---
# Étape 6 — Lire les erreurs du meilleur modèle

🎯 Les chiffres disent **combien** d'erreurs, pas **lesquelles**. Afficher les faux positifs et les faux négatifs du meilleur modèle (première ligne du tableau), avec leur probabilité, et les **lire**.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df[(df["label"] == 0) & (df["pred"] == 1)]` | faux positifs (`&` = « et », parenthèses obligatoires) | — |
| `.sort_values("proba")` | du moins sûr au plus sûr | — |

In [ ]:
# ✍️ À toi
# 1. meilleur = la cle de la première ligne de tableau ; df = predictions[meilleur]
# 2. faux_positifs = les ham prédits spam ; faux_negatifs = les spams prédits ham
# 3. affiche-les (text et proba), triés par proba

In [ ]:
# ✅ Contrôle
m = matrices[meilleur]
assert len(faux_positifs) == m[0, 1] and len(faux_negatifs) == m[1, 0]
print("✅ étape 6")

✍️ **À écrire (2-3 phrases)** : qu'ont en commun les spams qui passent ? Et les SMS légitimes bloqués ? Certaines erreurs sont-elles compréhensibles, voire discutables (un « vrai ham » qui ressemble beaucoup à une pub) ?

*Ta réponse :*

---
# Étape 7 — Les deux modèles se trompent-ils sur les mêmes SMS ?

🎯 Compter les SMS de test mal classés par : les **deux** modèles, **seulement** le modèle simple, **seulement** DistilBERT.

Les deux fichiers de prédictions suivent l'ordre de `test.csv` : la ligne *i* est le même SMS dans les deux, on peut donc comparer ligne à ligne.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df["pred"] != df["label"]` | `True` pour chaque erreur | — |
| `a & b`, `a & ~b` | et / et pas | `~` = « non » |
| `.values` | compare par position, sans se soucier de l'index | — |

In [ ]:
# ✍️ À toi
# 1. erreurs_emb, erreurs_tr = les séries d'erreurs (True/False) de chaque modèle
# 2. n_les_deux, n_seulement_emb, n_seulement_tr = les trois comptes ; affiche-les
# 3. affiche les SMS ratés par les deux modèles

In [ ]:
# ✅ Contrôle
assert (predictions["embeddings"]["label"].values == predictions["transfer"]["label"].values).all(), "les deux fichiers ne sont pas dans le même ordre"
assert n_les_deux + n_seulement_emb == matrices["embeddings"][0, 1] + matrices["embeddings"][1, 0]
assert n_les_deux + n_seulement_tr == matrices["transfer"][0, 1] + matrices["transfer"][1, 0]
print("✅ étape 7")

✍️ **À écrire** : les erreurs se recouvrent-elles beaucoup ? Si elles sont **différentes**, une piste d'amélioration serait de **combiner** les deux modèles (moyenne des probabilités). Attention : pour l'évaluer proprement, il faudrait régler cette combinaison sur val, pas sur test.

*Ta réponse :*

---
# Étape 8 — Le tableau pour le README

🎯 Le README du projet contient un tableau « Résultats » à remplir. Le générer en **Markdown** pour le copier-coller, au même format :

```
| Modèle | Accuracy | Précision spam | Recall spam | F1 spam |
|---|---|---|---|---|
| ... | 0.987 | ... | ... | ... |
```

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `for _, ligne in tableau.iterrows():` | parcourt les lignes | — |
| `f"{nombre:.3f}"` | un nombre à 3 décimales | — |
| `"\n".join(liste_de_lignes)` | colle les lignes avec un retour à la ligne | — |

In [ ]:
# ✍️ À toi
# 1. lignes_md = l'en-tête, la ligne de tirets, puis une ligne par modèle
# 2. tableau_md = les lignes collées ; affiche-le avec print

In [ ]:
# ✅ Contrôle
assert tableau_md.count("\n") == len(tableau) + 1, "en-tête + tirets + une ligne par modèle"
assert all(m in tableau_md for m in tableau["modele"])
print("✅ étape 8 — copie le tableau ci-dessus dans le README")

---
# Étape 9 — Conclusion et recommandation

✍️ **À écrire** : c'est la partie que le jury retiendra. Quelques lignes pour chaque point.

**1. Performance atteinte** (la phrase exigée par l'énoncé) : quel modèle, quel F1, quel recall, quelle précision **sur test**.

*Ta réponse :*

**2. Recommandation pour AT&T** : quel modèle mettre en production, et pourquoi ? Pèse le gain de F1 (étape 3) contre le coût (étape 5) et la gravité des erreurs (étape 4).

*Ta réponse :*

**3. Limites** : ce qui rend ces chiffres moins sûrs qu'ils n'en ont l'air. Pistes : taille du test (98 spams seulement : une erreur de plus fait bouger le recall d'un point), données anciennes et britanniques, caractères abîmés de la source, quasi-doublons à un espace près, spams qui évoluent avec le temps...

*Ta réponse :*

**4. Pistes d'amélioration** : combinaison des modèles, validation croisée, modèle multilingue, données plus récentes...

*Ta réponse :*

---
# 📝 Check-list du livrable

| Exigence de l'énoncé | Où | Fait ? |
|---|---|---|
| un notebook qui fait le **preprocessing** | template 1 (`02-...`) | |
| ... et qui **entraîne un ou plusieurs modèles de deep learning** | templates 2 et 3 (`03-...`, `04-...`) | |
| conseil « start simple » | template 2 | |
| conseil « transfer learning » | template 3 | |
| **performance annoncée clairement** | ce notebook, étapes 3 et 9 | |
| tableau des résultats dans le README | étape 8 | |
| `scores/` poussé sur GitHub (pas `models/`) | `git add scores/` | |